In [23]:
%pip install geopandas shapely fiona pyarrow mercantile tqdm mapbox-vector-tile

Note: you may need to restart the kernel to use updated packages.


In [24]:
from pathlib import Path
import re
import warnings
from collections import defaultdict

import geopandas as gpd
import mercantile
import mapbox_vector_tile
from shapely.geometry import box, mapping
from shapely.validation import make_valid
from tqdm.auto import tqdm

In [25]:
# --------------------------------------------------
# USER SETTINGS
# --------------------------------------------------

INPUT_ROOT = Path(r"C:\Users\wilsonschutterj\data\geojson")

OUTPUT_ROOT = Path(r"C:\Users\wilsonschutterj\data\vector_tiles")

YEARS = range(2010, 2025)
FILETYPES = ["BG", "TRACT"]

# Start conservative. You can increase later.
MIN_ZOOM = 0
MAX_ZOOM = 10

# Keep tiles small.
# GEOID, year, geography_type, and statefp are always kept.
# Add columns here only if you really need them in the tile properties.
EXTRA_KEEP_COLUMNS = []

# Vector tile layer names used later in Mapbox GL JS as source-layer.
LAYER_NAMES = {
    "BG": "census_bg",
    "TRACT": "census_tract"
}

# Prefer parquet when both parquet and geojson exist.
PREFER_PARQUET = True

# Skip completely empty tiles.
SKIP_EMPTY_TILES = True

In [26]:
print("Input root:", INPUT_ROOT)
print("Input exists:", INPUT_ROOT.exists())

print("Output root:", OUTPUT_ROOT)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("\nSample folders:")
for p in list(INPUT_ROOT.iterdir())[:5]:
    print(" ", p)

Input root: C:\Users\wilsonschutterj\data\geojson
Input exists: True
Output root: C:\Users\wilsonschutterj\data\vector_tiles

Sample folders:
  C:\Users\wilsonschutterj\data\geojson\2010
  C:\Users\wilsonschutterj\data\geojson\2011
  C:\Users\wilsonschutterj\data\geojson\2012
  C:\Users\wilsonschutterj\data\geojson\2013
  C:\Users\wilsonschutterj\data\geojson\2014


In [27]:
def find_input_files(year, filetype):
    """
    Finds files inside:
      C:/Users/wilsonschutterj/data/geojson/YEAR/BG
      C:/Users/wilsonschutterj/data/geojson/YEAR/TRACT

    Examples:
      tl_2010_01_bg10.geojson
      tl_2010_01_bg10.parquet
      tl_2011_01_bg.geojson
      tl_2011_01_bg.parquet
      tl_2011_01_tract.geojson
      tl_2011_01_tract.parquet

    If both parquet and geojson exist for the same stem, keeps parquet only.
    """
    folder = INPUT_ROOT / str(year) / filetype.upper()

    if not folder.exists():
        print(f"Missing folder: {folder}")
        return []

    parquet_files = sorted(folder.glob("*.parquet"))
    geojson_files = sorted(folder.glob("*.geojson"))

    if PREFER_PARQUET:
        files_by_stem = {}

        for f in geojson_files:
            files_by_stem[f.stem] = f

        for f in parquet_files:
            files_by_stem[f.stem] = f

        return sorted(files_by_stem.values())

    else:
        files_by_stem = {}

        for f in parquet_files:
            files_by_stem[f.stem] = f

        for f in geojson_files:
            files_by_stem[f.stem] = f

        return sorted(files_by_stem.values())


def state_fips_from_name(path):
    """
    Extracts state FIPS from:
      tl_2010_01_bg10.parquet
      tl_2011_01_bg.parquet
      tl_2024_45_tract.geojson
    """
    match = re.search(r"tl_\d{4}_(\d{2})_", Path(path).name)

    if not match:
        return "unknown"

    return match.group(1)

In [28]:
for year in [2010, 2011, 2024]:
    for filetype in ["BG", "TRACT"]:
        files = find_input_files(year, filetype)
        print(f"{year} {filetype}: found {len(files)} files")
        for f in files[:3]:
            print("  ", f.name)
        print()

2010 BG: found 56 files
   tl_2010_01_bg10.parquet
   tl_2010_02_bg10.parquet
   tl_2010_04_bg10.parquet

2010 TRACT: found 552 files
   tl_2010_01001_tract10.parquet
   tl_2010_01003_tract10.parquet
   tl_2010_01005_tract10.parquet

2011 BG: found 55 files
   tl_2011_01_bg.parquet
   tl_2011_02_bg.parquet
   tl_2011_04_bg.parquet

2011 TRACT: found 56 files
   tl_2011_01_tract.parquet
   tl_2011_02_tract.parquet
   tl_2011_04_tract.parquet

2024 BG: found 56 files
   tl_2024_01_bg.parquet
   tl_2024_02_bg.parquet
   tl_2024_04_bg.parquet

2024 TRACT: found 56 files
   tl_2024_01_tract.parquet
   tl_2024_02_tract.parquet
   tl_2024_04_tract.parquet



In [29]:
def detect_geoid_column(gdf, year):
    """
    Detects Census GEOID field across 2010 and later TIGER files.
    Common possibilities:
      GEOID
      GEOID10
      GEOID20
      GEOIDFQ
      geoid
      geoid10
    """
    cols = list(gdf.columns)
    lower_map = {c.lower(): c for c in cols}

    candidates = [
        "geoid",
        f"geoid{str(year)[-2:]}",
        "geoid10",
        "geoid20",
        "geoid00",
        "geoidfq"
    ]

    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]

    for c in cols:
        if "geoid" in c.lower():
            return c

    raise ValueError(
        "No GEOID-like column found. "
        f"Available columns are: {cols}"
    )


def detect_state_column(gdf):
    """
    Detects state FIPS column when available.
    """
    cols = list(gdf.columns)
    lower_map = {c.lower(): c for c in cols}

    candidates = [
        "statefp",
        "statefp10",
        "statefp20"
    ]

    for candidate in candidates:
        if candidate in lower_map:
            return lower_map[candidate]

    return None


def standardize_gdf(gdf, year, filetype, fallback_statefp=None):
    """
    Standardizes geometry and properties before vector tile creation.

    Output properties preserved:
      geoid
      year
      geography_type
      statefp
      any EXTRA_KEEP_COLUMNS
    """
    if gdf.empty:
        return gdf

    if gdf.crs is None:
        warnings.warn("Input CRS is missing. Assuming EPSG:4326.")
        gdf = gdf.set_crs("EPSG:4326")

    if gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs("EPSG:4326")

    geoid_col = detect_geoid_column(gdf, year)

    gdf["geoid"] = gdf[geoid_col].astype(str)
    gdf["year"] = int(year)
    gdf["geography_type"] = filetype.upper()

    state_col = detect_state_column(gdf)

    if state_col:
        gdf["statefp"] = gdf[state_col].astype(str).str.zfill(2)
    elif fallback_statefp:
        gdf["statefp"] = str(fallback_statefp).zfill(2)
    else:
        gdf["statefp"] = gdf["geoid"].str.slice(0, 2)

    keep_cols = [
        "geometry",
        "geoid",
        "year",
        "geography_type",
        "statefp"
    ]

    for c in EXTRA_KEEP_COLUMNS:
        if c in gdf.columns and c not in keep_cols:
            keep_cols.append(c)

    gdf = gdf[keep_cols].copy()

    # Fix invalid geometries where possible.
    def fix_geometry(geom):
        if geom is None:
            return None

        if geom.is_empty:
            return None

        if not geom.is_valid:
            return make_valid(geom)

        return geom

    gdf["geometry"] = gdf.geometry.apply(fix_geometry)

    gdf = gdf[~gdf.geometry.isna()]
    gdf = gdf[~gdf.geometry.is_empty]

    return gdf

In [30]:
def tiles_for_gdf(gdf, min_zoom, max_zoom):
    """
    Returns XYZ tiles intersecting the total bounds of a GeoDataFrame.
    """
    west, south, east, north = gdf.total_bounds

    # Clamp to valid Web Mercator latitude range.
    west = max(float(west), -180)
    east = min(float(east), 180)
    south = max(float(south), -85.05112878)
    north = min(float(north), 85.05112878)

    all_tiles = []

    for z in range(min_zoom, max_zoom + 1):
        all_tiles.extend(
            list(
                mercantile.tiles(
                    west,
                    south,
                    east,
                    north,
                    z
                )
            )
        )

    return all_tiles


def encode_tile(features, layer_name, tile_bounds):
    """
    Encodes a list of GeoJSON-like features into a Mapbox Vector Tile.

    tile_bounds should be:
      west, south, east, north
    """
    layer = {
        "name": layer_name,
        "features": features
    }

    pbf = mapbox_vector_tile.encode(
        [layer],
        default_options={
            "quantize_bounds": tile_bounds,
            "extents": 4096
        }
    )

    return pbf


def get_candidate_indices(gdf, tile_bbox):
    """
    Uses spatial index when available.
    Includes a slower fallback.
    """
    try:
        sindex = gdf.sindex
        return list(sindex.query(tile_bbox, predicate="intersects"))
    except Exception:
        return list(gdf[gdf.intersects(tile_bbox)].index)


In [ ]:
from pathlib import Path

def get_output_base(year, filetype, statefp):
    return (
        OUTPUT_ROOT /
        str(year) /
        filetype.upper() /
        str(statefp).zfill(2)
    )


def get_completion_file(year, filetype, statefp):
    return (
        get_output_base(year, filetype, statefp) /
        "_COMPLETE.txt"
    )


def already_processed(year, filetype, statefp):
    """
    Returns True only if a completion marker exists.
    """

    completion_file = get_completion_file(
        year,
        filetype,
        statefp
    )

    return completion_file.exists()


def mark_complete(year, filetype, statefp,
                  features, tiles_written):
    """
    Create completion marker after a successful run.
    """

    completion_file = get_completion_file(
        year,
        filetype,
        statefp
    )

    completion_file.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    completion_file.write_text(
        (
            f"year={year}\n"
            f"filetype={filetype}\n"
            f"statefp={statefp}\n"
            f"features={features}\n"
            f"tiles_written={tiles_written}\n"
        ),
        encoding="utf-8"
    )

In [31]:
def write_vector_tiles_for_file(path, year, filetype):
    """
    Converts one state/year/geography file into XYZ .pbf vector tiles.

    Output:
      C:/Users/wilsonschutterj/data/vector_tiles/YEAR/FILETYPE/STATEFP/z/x/y.pbf
    """
    path = Path(path)

    statefp = state_fips_from_name(path)
    layer_name = LAYER_NAMES[filetype.upper()]

    out_base = OUTPUT_ROOT / str(year) / filetype.upper() / statefp

    print(f"\nReading: {path}")
    print(f"Output base: {out_base}")

    gdf = read_spatial_file(path, year, filetype)

    if gdf.empty:
        print(f"Skipping empty file: {path}")
        return {
            "path": str(path),
            "year": year,
            "filetype": filetype,
            "statefp": statefp,
            "features": 0,
            "tiles_written": 0
        }

    print(f"Features: {len(gdf):,}")
    print(f"Columns kept: {list(gdf.columns)}")
    print(f"Bounds: {gdf.total_bounds}")

    tile_list = tiles_for_gdf(gdf, MIN_ZOOM, MAX_ZOOM)

    tiles_written = 0

    for tile in tqdm(tile_list, desc=f"{year} {filetype} {statefp}", leave=False):
        bounds = mercantile.bounds(tile)

        tile_bbox = box(
            bounds.west,
            bounds.south,
            bounds.east,
            bounds.north
        )

        tile_bounds = (
            bounds.west,
            bounds.south,
            bounds.east,
            bounds.north
        )

        candidate_indices = get_candidate_indices(gdf, tile_bbox)

        if not candidate_indices:
            continue

        features = []

        for idx in candidate_indices:
            row = gdf.iloc[idx]
            geom = row.geometry

            if geom is None or geom.is_empty:
                continue

            if not geom.intersects(tile_bbox):
                continue

            try:
                clipped = geom.intersection(tile_bbox)
            except Exception:
                fixed = make_valid(geom)
                clipped = fixed.intersection(tile_bbox)

            if clipped is None or clipped.is_empty:
                continue

            props = {
                "geoid": str(row["geoid"]),
                "year": int(row["year"]),
                "geography_type": str(row["geography_type"]),
                "statefp": str(row["statefp"]).zfill(2)
            }

            for c in EXTRA_KEEP_COLUMNS:
                if c in row.index:
                    val = row[c]
                    if val is not None:
                        props[c] = val

            features.append(
                {
                    "geometry": mapping(clipped),
                    "properties": props
                }
            )

        if SKIP_EMPTY_TILES and not features:
            continue

        try:
            pbf = encode_tile(
                features=features,
                layer_name=layer_name,
                tile_bounds=tile_bounds
            )
        except Exception as e:
            print(f"Tile encode failed for {tile}: {e}")
            continue

        out_path = out_base / str(tile.z) / str(tile.x) / f"{tile.y}.pbf"
        out_path.parent.mkdir(parents=True, exist_ok=True)

        with open(out_path, "wb") as f:
            f.write(pbf)

        tiles_written += 1

    print(f"Tiles written: {tiles_written:,}")

    return {
        "path": str(path),
        "year": year,
        "filetype": filetype,
        "statefp": statefp,
        "features": len(gdf),
        "tiles_written": tiles_written
    }

In [32]:
def run_all():
    results = []

    for year in YEARS:
        for filetype in FILETYPES:
            files = find_input_files(year, filetype)

            print("\n" + "=" * 70)
            print(f"{year} {filetype}: found {len(files)} files")
            print("=" * 70)

            for path in files:
                try:
                    result = write_vector_tiles_for_file(
                        path=path,
                        year=year,
                        filetype=filetype
                    )

                    results.append(result)

                    print(
                        f"Done: year={result['year']}, "
                        f"type={result['filetype']}, "
                        f"state={result['statefp']}, "
                        f"features={result['features']:,}, "
                        f"tiles={result['tiles_written']:,}"
                    )

                except Exception as e:
                    print(f"\nFAILED: {path}")
                    print(f"Error: {e}")

                    results.append(
                        {
                            "path": str(path),
                            "year": year,
                            "filetype": filetype,
                            "statefp": state_fips_from_name(path),
                            "features": None,
                            "tiles_written": None,
                            "error": str(e)
                        }
                    )

    return results

In [33]:
results = run_all()


2010 BG: found 56 files

Reading: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_01_bg10.parquet
Output base: C:\Users\wilsonschutterj\data\vector_tiles\2010\BG\01

FAILED: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_01_bg10.parquet
Error: name 'read_spatial_file' is not defined

Reading: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_02_bg10.parquet
Output base: C:\Users\wilsonschutterj\data\vector_tiles\2010\BG\02

FAILED: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_02_bg10.parquet
Error: name 'read_spatial_file' is not defined

Reading: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_04_bg10.parquet
Output base: C:\Users\wilsonschutterj\data\vector_tiles\2010\BG\04

FAILED: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_04_bg10.parquet
Error: name 'read_spatial_file' is not defined

Reading: C:\Users\wilsonschutterj\data\geojson\2010\BG\tl_2010_05_bg10.parquet
Output base: C:\Users\wilsonschutterj\data\vector_tiles\2010\BG\05

FAILED: 